# Phân cụm khách hàng (Customer Clustering) và Chuẩn bị dữ liệu Causal ML
Tài liệu này bao gồm toàn bộ quá trình:
1. Đọc và chuẩn hóa dữ liệu.
2. Tính toán RFM & Nhân khẩu học.
3. K-Means Clustering (Elbow & Silhouette).
4. Phác họa đặc điểm các cụm (Profiling) & Trực quan hóa.
5. Xuất dữ liệu đã gán nhãn cho mô hình Causal ML.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

# Cấu hình matplotlib hiển thị trực tiếp trong notebook
%matplotlib inline


## 1. Đọc và Xử lý dữ liệu ban đầu
Tính toán RFM từ `transaction_data.csv` và ghép nối với `hh_demographic.csv`.


In [ ]:
print("Đọc dữ liệu...")
trans_df = pd.read_csv("data/transaction_data.csv")
demo_df = pd.read_csv("data/hh_demographic.csv")

# Tính RFM
current_day = trans_df['DAY'].max()
rfm = trans_df.groupby('household_key').agg(
    Recency=('DAY', lambda x: current_day - x.max()),
    Frequency=('BASKET_ID', 'nunique'),
    Monetary=('SALES_VALUE', 'sum')
).reset_index()

# Đổi tên cột demographic
demo_df.columns = ['Age_Desc', 'Marital_Status', 'Income_Desc', 'Homeowner_Desc', 'HH_Comp_Desc', 'Household_Size_Desc', 'Kid_Category_Desc', 'household_key']

# Merge
merged_df = pd.merge(rfm, demo_df, on='household_key', how='inner')

# Mã hóa
income_mapping = {
    'Under 15K': 1, '15-24K': 2, '25-34K': 3, '35-49K': 4, 
    '50-74K': 5, '75-99K': 6, '100-124K': 7, '125-149K': 8, 
    '150-174K': 9, '175-199K': 10, '200-249K': 11, '250K+': 12
}
merged_df['Income_Level'] = merged_df['Income_Desc'].map(income_mapping)
if merged_df['Income_Level'].isnull().all():
    level_mapping = {f'Level{i}': i for i in range(1, 13)}
    merged_df['Income_Level'] = merged_df['Income_Desc'].map(level_mapping)

size_mapping = {'1': 1, '2': 2, '3': 3, '4': 4, '5+': 5}
merged_df['Household_Size'] = merged_df['Household_Size_Desc'].astype(str).map(size_mapping).fillna(1)

age_mapping = {'19-24': 1, '25-34': 2, '35-44': 3, '45-54': 4, '55-64': 5, '65+': 6}
merged_df['Age_Level'] = merged_df['Age_Desc'].map(age_mapping)
if merged_df['Age_Level'].isnull().all():
    age_group_mapping = {f'Age Group{i}': i for i in range(1, 7)}
    merged_df['Age_Level'] = merged_df['Age_Desc'].map(age_group_mapping)

features_cols = ['Recency', 'Frequency', 'Monetary', 'Income_Level', 'Age_Level', 'Household_Size']
features = merged_df[features_cols].dropna()
final_df = merged_df.loc[features.index].copy()

print(f"Số lượng khách hàng đủ dữ kiện phân tích: {len(features)}")


## 2. Tìm K tối ưu bằng Elbow & Silhouette


In [ ]:
# Chuẩn hóa
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

wcss = []
silhouette_scores = []
K_range = range(2, 11)

print("Tính toán Elbow và Silhouette...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, cluster_labels))

# Trực quan hóa
fig, ax1 = plt.subplots(figsize=(10, 6))
color1 = 'tab:blue'
ax1.set_xlabel('Số cụm (k)')
ax1.set_ylabel('WCSS', color=color1)
line1 = ax1.plot(K_range, wcss, marker='o', linestyle='-', color=color1, label='WCSS')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(K_range)
ax1.grid(True, linestyle='--', alpha=0.6)

ax2 = ax1.twinx()  
color2 = 'tab:orange'
ax2.set_ylabel('Silhouette Score', color=color2)
line2 = ax2.plot(K_range, silhouette_scores, marker='s', linestyle='-', color=color2, label='Silhouette')
ax2.tick_params(axis='y', labelcolor=color2)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')
plt.title('Kết hợp Elbow Method và Silhouette Score')
plt.show()

best_k = K_range[np.argmax(silhouette_scores)]
print(f"Số cụm tối ưu được chọn: k = {best_k}")


## 3. Thực hiện K-Means & Profiling


In [ ]:
# Chạy K-Means với K tốt nhất
kmeans_opt = KMeans(n_clusters=best_k, random_state=42, n_init=10)
final_df['Cluster'] = kmeans_opt.fit_predict(X_scaled)

print("--- Kích thước mỗi cụm ---")
cluster_sizes = final_df['Cluster'].value_counts().sort_index()
cluster_props = final_df['Cluster'].value_counts(normalize=True).sort_index() * 100
size_df = pd.DataFrame({'Số lượng': cluster_sizes, 'Tỷ lệ (%)': cluster_props})
display(size_df)

print("\n--- Đặc điểm trung bình của mỗi cụm ---")
profile = final_df.groupby('Cluster')[features_cols].mean().round(2)
display(profile)


## Trực quan hóa RFM 3D (3D Plot)
Biểu đồ 3D thể hiện sự phân bổ của khách hàng theo 3 trục Recency, Frequency, và Monetary.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

xs = final_df['Recency']
ys = final_df['Frequency']
zs = final_df['Monetary']
clusters_ids = final_df['Cluster']

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

for i in sorted(clusters_ids.unique()):
    ax.scatter(
        xs[clusters_ids == i], 
        ys[clusters_ids == i], 
        zs[clusters_ids == i], 
        c=colors[i % len(colors)], 
        s=30, 
        label=f'Cluster {i}', 
        alpha=0.6,
        edgecolor='w'
    )

ax.set_title('RFM Customer Segmentation - 3D Visualization', fontsize=15, fontweight='bold')
ax.set_xlabel('Recency')
ax.set_ylabel('Frequency')
ax.set_zlabel('Monetary')
ax.legend(title="Clusters")

plt.tight_layout()
plt.show()

## 4. Trực quan hoá sự phân tách (PCA)


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
final_df['PCA1'] = X_pca[:, 0]
final_df['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(10, 8))
clusters = sorted(final_df['Cluster'].unique())
for c in clusters:
    subset = final_df[final_df['Cluster'] == c]
    plt.scatter(subset['PCA1'], subset['PCA2'], label=f'Cluster {c}', alpha=0.6)
plt.legend()
plt.title('Sự tách biệt cụm trong không gian PCA 2D')
plt.show()


## 5. Đặt tên Cụm & Vẽ Boxplots


In [ ]:
# Naming
cluster_names = {}
overall_monetary_mean = final_df['Monetary'].mean()
overall_recency_mean = final_df['Recency'].mean()

for cluster_id, row in profile.iterrows():
    if row['Monetary'] > overall_monetary_mean * 1.5:
        name = "High-Value Loyal"
    elif row['Recency'] > overall_recency_mean * 1.5:
        name = "Churned/At-Risk"
    elif row['Frequency'] > final_df['Frequency'].mean():
        name = "Frequent Shoppers"
    else:
        name = "Standard Customers"
    
    if row['Income_Level'] > final_df['Income_Level'].mean():
        name += " (High Income)"
    
    cluster_names[cluster_id] = f"Cluster {cluster_id}: {name}"

final_df['Cluster_Name'] = final_df['Cluster'].map(lambda x: cluster_names[x].split(': ')[1])
print("Tên các cụm đã gán:")
for k, v in cluster_names.items():
    print(v)

# Boxplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Phân phối các đặc điểm theo từng Cụm', fontsize=16)
for i, feature in enumerate(features_cols):
    ax = axes[i // 3, i % 3]
    grouped_data = [final_df[final_df['Cluster'] == c][feature] for c in clusters]
    ax.boxplot(grouped_data)
    ax.set_xticks(range(1, len(clusters) + 1))
    ax.set_xticklabels([f"C{c}" for c in clusters])
    ax.set_title(feature)
plt.tight_layout()
plt.show()


## 6. Xuất Dữ liệu cho Causal ML


In [ ]:
causal_df = pd.get_dummies(final_df, columns=['Cluster_Name'], drop_first=False)

output_file = "causal_ml_ready_data.csv"
causal_df.to_csv(output_file, index=False)
print(f"Đã lưu dataset cuối cùng vào '{output_file}', sẵn sàng cho Causal ML.")
